# HRP Database Setup

This notebook does the following:
1. Loading health monitoring data from Excel files
2. Setting up a local SQLite database for the data storage
3. Creating a normalized schema out of measurements, seniors, and other tables.

**Summary of Results:**
- **Seniors**: 14,170 unique seniors
- **Measurements**: 164,331,155 measurements for different types
- **Medical Info**: 8,739 seniors with disease & medication data
- **Diseases**: 161 unique diseases
- **Medications**: 1,731 unique medications
- **SOS Alerts**: 8,983 alert records
- **Storage**: Local SQLite database

In [28]:
import sys
import os
import time
import sqlite3
from pathlib import Path
import warnings

import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.components.load_data import load_all_data

warnings.filterwarnings('ignore')

## Section 1: Load and Inspect Raw Excel Data

Load the new collection: twelve measurement files, medical/diseases, seniors demographics (gender, birthdate, age), and SOS alerts. Inspect structure, types, and basic quality.

In [29]:
old_data_dir = Path("../data/raw/HRP_old")
new_data_dir = Path("../data/raw/2026-02-08")

measurement_files = [
    # Old 30-day data (Nov)
    old_data_dir / "data_202512221122-01-09.xlsx",
    old_data_dir / "data_202511181045.xlsx",
    old_data_dir / "data_202512221231-16-23.xlsx",
    old_data_dir / "data_202512221344-24-30.xlsx",
    
    # New 60-day data (Dec - Jan)
    new_data_dir / "data-2025-12-01-07_202602091218.xlsx",
    new_data_dir / "data-2025-12-08-14_202602091252.xlsx",
    new_data_dir / "data-2025-12-15-21_202602091330.xlsx",
    new_data_dir / "data-2025-12-22-28_202602091430.xlsx",
    new_data_dir / "data-2025-12-29-2026-01-04_202602091614.xlsx",
    new_data_dir / "data-2026-01-05-11-_202602092056.xlsx",
    new_data_dir / "data-2026-01-12-18_202602100141.xlsx",
    new_data_dir / "data-2026-01-19-25_202602100226.xlsx",
    new_data_dir / "data-2026-01-26-31-.xlsx",
]

for p in measurement_files:
    assert p.exists(), f"Missing file: {p}"

In [30]:
demo_old = pd.read_excel(old_data_dir / "SeniorGenderAge_202512221409.xlsx")
demo_new = pd.read_excel(new_data_dir / "SeniorGenderAge_202602082214.xlsx")

demo_combined = pd.concat([demo_old, demo_new])
df_demo_raw = demo_combined.drop_duplicates(subset=['seniorID'], keep='last').reset_index(drop=True)

In [31]:
meds_old = pd.read_excel(old_data_dir / "Med&Diseases_202512221410.xlsx", engine="openpyxl")
meds_new = pd.read_excel(new_data_dir / "Med&Diseases_202602082215.xlsx", engine="openpyxl")

meds_combined = pd.concat([meds_old, meds_new])
df_medical = meds_combined.drop_duplicates().reset_index(drop=True)

In [32]:
sos_old = pd.read_excel(old_data_dir / "SOS_202512221411.xlsx")
sos_new = pd.read_excel(new_data_dir / "SOS_202602082216.xlsx")

sos_combined = pd.concat([sos_old, sos_new])
df_sos = sos_combined.drop_duplicates().reset_index(drop=True)

In [33]:
df_medical.shape

(9133, 3)

In [34]:
df_medical.columns

Index(['seniorID', 'diseaseNames', 'medicineNames'], dtype='object')

In [35]:
df_medical.dtypes

seniorID          int64
diseaseNames     object
medicineNames    object
dtype: object

In [36]:
df_medical.head()

,seniorID,diseaseNames,medicineNames
0,2875,"Osteoporoza,Nadciśnienie tętnicze,Arytmia serc...","Acard,Emanera,Agen,Concor,Valzek"
1,3755,"Miażdzyca,Osteoporoza","Gensulin,Beto,Furosemidum,Amlopin,Zahron,Berod..."
2,3762,"Cukrzyca,Niedoczynnośc tarczycy,Niedoczynnośc ...","Letrox,Diosminex,Valsacor,Metformax,Bibloc,Pol..."
3,3805,"Stomia,Niedosłuch,Skolioza","Pregabalin,Staveran,Neurovit"
4,4367,"Miażdżyca kończyn dolnych,Niewydolnośc układu ...","Allupol,Cipropol,Eliquis,Ezehron,Areplex"


In [37]:
df_medical.isnull().sum()

seniorID         0
diseaseNames     0
medicineNames    0
dtype: int64

In [38]:
df_demo_raw.shape

(14893, 4)

In [39]:
df_demo_raw.columns

Index(['seniorID', 'gender', 'birthDate', 'age'], dtype='object')

In [40]:
df_demo_raw.dtypes

seniorID       int64
gender        object
birthDate     object
age          float64
dtype: object

In [41]:
df_demo_raw.isnull().sum()

seniorID       0
gender         0
birthDate    339
age          339
dtype: int64

In [42]:
df_sos.shape

(14392, 3)

In [43]:
df_sos.columns

Index(['seniorID', 'alertDate', 'sosNote'], dtype='object')

In [44]:
df_sos.dtypes

seniorID              int64
alertDate    datetime64[ns]
sosNote              object
dtype: object

In [45]:
df_sos.head()

,seniorID,alertDate,sosNote
0,3205,2025-11-30 17:07:51,Alarm przypadkowy
1,3221,2025-11-25 19:38:36,Alarm przypadkowy
2,3275,2025-11-14 15:01:34,Alarm przypadkowy
3,3279,2025-11-09 11:58:31,Alarm przypadkowy
4,3283,2025-11-17 18:44:32,Alarm przypadkowy


In [46]:
df_sos.isnull().sum()

seniorID       0
alertDate      0
sosNote      127
dtype: int64

In [47]:
xls = pd.ExcelFile(measurement_files[0])
sheet_names = xls.sheet_names
print(f"Total sheets in first file: {len(sheet_names)}")
print(f"Sheet names: {sheet_names}\n")

Total sheets in first file: 26
Sheet names: ['expdata', 'expdata#1', 'expdata#2', 'expdata#3', 'expdata#4', 'expdata#5', 'expdata#6', 'expdata#7', 'expdata#8', 'expdata#9', 'expdata#10', 'expdata#11', 'expdata#12', 'expdata#13', 'expdata#14', 'expdata#15', 'expdata#16', 'expdata#17', 'expdata#18', 'expdata#19', 'expdata#20', 'expdata#21', 'expdata#22', 'expdata#23', 'expdata#24', 'expdata#25']



In [48]:
for i, sheet_name in enumerate(sheet_names[:2]):
    df = pd.read_excel(measurement_files[0], sheet_name=sheet_name, nrows=5, engine="openpyxl")
    print(f"\nSheet '{sheet_name}':")
    print(f"    Col 0 (seniorID): {df.iloc[:, 0].values[:2]}")
    print(f"    Col 1 (value): {df.iloc[:, 1].values[:2]}")
    print(f"    Col 2 (sbp): {df.iloc[:, 2].values[:2]}")
    print(f"    Col 3 (dbp): {df.iloc[:, 3].values[:2]}")
    print(f"    Col 4 (date): {df.iloc[:, 4].values[:2]}")
    print(f"    Col 5 (type): {df.iloc[:, 5].values[:2]}")


Sheet 'expdata':
    Col 0 (seniorID): [48129 48427]
    Col 1 (value): [36.3 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-03T23:16:12.000000000' '2025-11-03T23:16:12.000000000']
    Col 5 (type): ['Temperature' 'Temperature']

Sheet 'expdata#1':
    Col 0 (seniorID): [48313 42183]
    Col 1 (value): [36.3 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-04T07:59:41.000000000' '2025-11-04T07:59:41.000000000']
    Col 5 (type): ['Temperature' 'Temperature']


## Section 2: Store Data in SQLite Database

Use the pipeline from `src.components.load_data` to load demographics, measurements, medical info, and alerts into SQLite.

In [57]:
# Run end-to-end load using the following pipeline
# Use streaming to avoid large memory usage and commit in batches
# Set fresh_start=True to rebuild the DB and process all sheets from scratch
load_all_data(fresh_start=True, streaming=True, batch_rows=100_000, resume=False)

INFO:src.components.database:Database initialized at c:\Users\eldar\Projects\AI-CVD\db\hrp_data.db
INFO:src.components.load_data:Loading seniors demographics from 1 files


INFO:src.components.load_data:Loaded 13317 unique senior demographic rows
INFO:src.components.load_data:Upserted 13317 seniors with demographics
INFO:src.components.load_data:Loading seniors demographics from 1 files
INFO:src.components.load_data:Loaded 14170 unique senior demographic rows
INFO:src.components.load_data:Upserted 14170 seniors with demographics
INFO:src.components.load_data:Streaming measurements from c:\Users\eldar\Projects\AI-CVD\data\raw\HRP_old\data_202512221122-01-09.xlsx
INFO:src.components.load_data:  expdata: +100,000 (total 100,000)
INFO:src.components.load_data:  expdata: +100,000 (total 200,000)
INFO:src.components.load_data:  expdata: +100,000 (total 300,000)
INFO:src.components.load_data:  expdata: +100,000 (total 400,000)
INFO:src.components.load_data:  expdata: +100,000 (total 500,000)
INFO:src.components.load_data:  expdata: +100,000 (total 600,000)
INFO:src.components.load_data:  expdata: +100,000 (total 700,000)
INFO:src.components.load_data:  expdata: 


DATABASE SUMMARY
seniors.......................          14,929
measurements..................     255,019,140
alerts........................          14,392
medical_info (raw)............           9,129
diseases......................             161
medicines.....................           1,765
senior_diseases...............          55,383
senior_medicines..............          53,852


In [58]:
db_path = Path("../db/hrp_data.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [ ]:
if "conn" in locals() and conn:
    try:
        conn.close()
        print("Closed prior database connection")
    except Exception as e:
        print(f"Warning while closing prior connection: {e}")

Closed prior database connection


In [59]:
print(f"Database initialized at {db_path.as_posix()}")
print(f"Database size: {db_path.stat().st_size / 1024:.1f} KB")

Database initialized at ../db/hrp_data.db
Database size: 50551004.0 KB


In [60]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\nTables created: {[t[0] for t in tables]}")


Tables created: ['seniors', 'measurements', 'sqlite_sequence', 'medical_info', 'diseases', 'medicines', 'senior_diseases', 'senior_medicines', 'alerts', 'ingestion_state']


# Section 3: Verify/Inspect Data Integrity

In [61]:
# Pull measurements 
df_measurements = pd.read_sql("SELECT * FROM measurements LIMIT 100000", conn)
print(df_measurements.shape)
df_measurements.head()

(100000, 7)


,id,senior_id,value,sbp,dbp,date,type
0,1,48129,36.3,None,None,2025-11-03 23:16:12,Temperature
1,2,48427,36.6,None,None,2025-11-03 23:16:12,Temperature
2,3,45310,36.3,None,None,2025-11-03 23:16:12,Temperature
3,4,29421,36.6,None,None,2025-11-03 23:16:12,Temperature
4,5,45316,36.9,None,None,2025-11-03 23:16:12,Temperature


In [62]:
# Medical information counts
counts_med = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM medical_info", conn)
counts_med

,n_rows,n_seniors
0,9129,9129


In [63]:
# Alerts counts
counts_alerts = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM alerts", conn)
counts_alerts

,n_rows,n_seniors
0,14392,6044


In [64]:
# Preview alerts
pd.read_sql("SELECT * FROM alerts LIMIT 5", conn)

,alert_id,senior_id,alert_date,sos_note
0,1,3205,2025-11-30 17:07:51,Alarm przypadkowy
1,2,3221,2025-11-25 19:38:36,Alarm przypadkowy
2,3,3275,2025-11-14 15:01:34,Alarm przypadkowy
3,4,3279,2025-11-09 11:58:31,Alarm przypadkowy
4,5,3283,2025-11-17 18:44:32,Alarm przypadkowy


In [65]:
# Measurement type distribution
measure_type_counts = pd.read_sql(
    "SELECT type, COUNT(*) AS cnt FROM measurements GROUP BY type ORDER BY cnt DESC",
    conn,
)
measure_type_counts

,type,cnt
0,Heartrate,57764491
1,BloodPressure,57764420
2,Temperature,57534646
3,Saturation,46283823
4,Steps,35671760


## Section 4: Query and Validate Stored Data

Execute SQL queries to retrieve data and perform basic analysis to confirm database functionality.

### Example 1: Get all measurements for a specific type

In [66]:
query1 = """
    SELECT senior_id, value, date, type
    FROM measurements
    WHERE type = 'Heartrate'
    ORDER BY date
    LIMIT 10
"""

In [67]:
df_example1 = pd.read_sql(query1, conn)
df_example1

,senior_id,value,date,type
0,20307,88.0,2025-11-01 00:00:10,Heartrate
1,8721,62.0,2025-11-01 00:00:10,Heartrate
2,41788,104.0,2025-11-01 00:00:10,Heartrate
3,44626,85.0,2025-11-01 00:00:10,Heartrate
4,44759,65.0,2025-11-01 00:00:10,Heartrate
5,44228,50.0,2025-11-01 00:00:10,Heartrate
6,43633,75.0,2025-11-01 00:00:10,Heartrate
7,30312,73.0,2025-11-01 00:00:10,Heartrate
8,28661,67.0,2025-11-01 00:00:10,Heartrate
9,32587,54.0,2025-11-01 00:00:11,Heartrate


### Example 2: Aggregate statistics by measurement type

In [68]:
query2 = """
    SELECT 
        type,
        COUNT(*) as measurement_count,
        COUNT(DISTINCT senior_id) as unique_seniors,
        AVG(value) as avg_value,
        MIN(value) as min_value,
        MAX(value) as max_value,
        ROUND(AVG(value), 2) as mean
    FROM measurements
    WHERE value IS NOT NULL
    GROUP BY type
    ORDER BY measurement_count DESC
"""

In [69]:
df_example2 = pd.read_sql(query2, conn)
df_example2

,type,measurement_count,unique_seniors,avg_value,min_value,max_value,mean
0,Heartrate,57764491,14755,73.305297,1.0,214.0,73.31
1,Temperature,57534646,14725,36.680368,36.2,127.9,36.68
2,Saturation,46283823,14694,97.055387,80.0,100.0,97.06
3,Steps,35671760,14228,2980.794772,1.0,50147.0,2980.79


### Example 3: Get measurements for a specific senior

In [70]:
sample_senior_id = int(df_measurements.iloc[0]["senior_id"])
query3 = """
    SELECT senior_id, value, sbp, dbp, date, type
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date DESC
    LIMIT 10
"""

In [71]:
df_example3 = pd.read_sql(query3, conn, params=[sample_senior_id])
df_example3

,senior_id,value,sbp,dbp,date,type
0,48129,98.0,NaN,NaN,2026-01-31 23:53:19,Saturation
1,48129,NaN,127.0,83.0,2026-01-31 23:53:19,BloodPressure
2,48129,74.0,NaN,NaN,2026-01-31 23:53:19,Heartrate
3,48129,36.6,NaN,NaN,2026-01-31 23:53:19,Temperature
4,48129,99.0,NaN,NaN,2026-01-31 23:43:19,Saturation
5,48129,NaN,130.0,78.0,2026-01-31 23:43:19,BloodPressure
6,48129,67.0,NaN,NaN,2026-01-31 23:43:19,Heartrate
7,48129,36.8,NaN,NaN,2026-01-31 23:43:19,Temperature
8,48129,95.0,NaN,NaN,2026-01-31 23:33:20,Saturation
9,48129,NaN,124.0,85.0,2026-01-31 23:33:20,BloodPressure


### Example 4: Query performance test

In [72]:
start = time.time()
query4 = "SELECT * FROM measurements WHERE type = 'Heartrate' LIMIT 1000"
df_example4 = pd.read_sql(query4, conn)
elapsed = time.time() - start

In [73]:
print(f"  Retrieved {len(df_example4)} rows in {elapsed:.4f} seconds")

  Retrieved 1000 rows in 0.0080 seconds


### Example 4: Blood Pressure Analysis

In [74]:
query5 = """
    SELECT senior_id, sbp, dbp, date, type
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
    ORDER BY date DESC
    LIMIT 10
"""

In [75]:
df_example5 = pd.read_sql(query5, conn)
df_example5

,senior_id,sbp,dbp,date,type
0,22282,135.0,64.0,2026-01-31 23:59:30,BloodPressure
1,13258,137.0,73.0,2026-01-31 23:59:30,BloodPressure
2,38877,125.0,70.0,2026-01-31 23:59:25,BloodPressure
3,53969,122.0,76.0,2026-01-31 23:59:23,BloodPressure
4,38965,112.0,74.0,2026-01-31 23:59:23,BloodPressure
5,53795,132.0,87.0,2026-01-31 23:59:23,BloodPressure
6,43762,146.0,81.0,2026-01-31 23:59:23,BloodPressure
7,48837,136.0,76.0,2026-01-31 23:59:23,BloodPressure
8,16457,135.0,76.0,2026-01-31 23:59:23,BloodPressure
9,53683,129.0,75.0,2026-01-31 23:59:23,BloodPressure


### Example 5: Get Blood Pressure Statistics

In [76]:
query5_stats = """
    SELECT 
        COUNT(*) as bp_measurements,
        COUNT(DISTINCT senior_id) as seniors_with_bp,
        AVG(sbp) as avg_systolic,
        AVG(dbp) as avg_diastolic,
        MIN(sbp) as min_systolic,
        MAX(sbp) as max_systolic,
        MIN(dbp) as min_diastolic,
        MAX(dbp) as max_diastolic
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
"""

In [77]:
df_bp_stats = pd.read_sql(query5_stats, conn)
df_bp_stats

,bp_measurements,seniors_with_bp,avg_systolic,avg_diastolic,min_systolic,max_systolic,min_diastolic,max_diastolic
0,57764420,14755,129.436936,78.605155,68.0,212.0,23.0,157.0
